[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# AI Program

## Deep Learning - Convolution Neural Network - Skip Connection Effect

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 02/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/SkipConnectionEffect.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np

# Scientific Python

# Image Processing & Computer Vision

# Machine Learning
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Python Library
import random
import time

# Typing 
from typing import Callable, Dict, List, Tuple
from numpy.typing import NDArray
from torch import Tensor

# Visualization
from matplotlib import cm
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

 ```python
 # You need to start writing
 ?????
 ```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)
torch.manual_seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

D_CLASSES = {ii: f'{ii}' for ii in range(10)}
L_CLASSES = list(D_CLASSES.keys())

In [ ]:
# Course Packages


In [ ]:
# Auxiliary Functions

class ConvBlock(nn.Module):
    def __init__(self, numChannels: int, useSkip: bool) -> None:
        super().__init__()

        self.useSkip = useSkip
        self.conv1   = nn.Conv2d(numChannels, numChannels, kernel_size = 3, padding = 1)
        self.conv2   = nn.Conv2d(numChannels, numChannels, kernel_size = 3, padding = 1)

    def forward(self, tX: Tensor) -> Tensor:

        tY = torch.tanh(self.conv1(tX))
        tY = self.conv2(tY)
        if self.useSkip:
            tY = tY + tX #<! The only difference between the two models

        return torch.tanh(tY)

class SimpleConvNet(nn.Module):
    def __init__(self, useSkip: bool, numBlocks: int = 6, numChannels: int = 16, numCls: int = 10) -> None:
        super().__init__()

        self.convIn  = nn.Conv2d(1, numChannels, kernel_size = 3, padding = 1)
        self.oBlocks = nn.Sequential(*[ConvBlock(numChannels, useSkip) for _ in range(numBlocks)])
        self.fcOut   = nn.Linear(numChannels, numCls)

    def forward(self, tX: Tensor) -> Tensor:

        tX = torch.tanh(self.convIn(tX))
        tX = self.oBlocks(tX)
        tX = torch.mean(tX, dim = (2, 3)) #<! Global average pooling

        return self.fcOut(tX)

def TrainModel(oModel: nn.Module, dlTrain: DataLoader, hL: Callable, numEpochs: int, learnRate: float, runDevice: torch.device) -> Tuple[nn.Module, List[float]]:

    oModel = oModel.to(runDevice)
    oOpt   = torch.optim.AdamW(oModel.parameters(), lr = learnRate, weight_decay = 1e-4)
    lTrainLoss = []

    for ii in range(numEpochs):
        oModel.train()
        epochLoss  = 0.0
        numSamples = 0
        for tX, vY in dlTrain:
            tX = tX.to(runDevice)
            vY = vY.to(runDevice)

            mZ      = oModel(tX)
            valLoss = hL(mZ, vY)

            oOpt.zero_grad()
            valLoss.backward()
            oOpt.step()

            epochLoss  += tX.shape[0] * valLoss.item()
            numSamples += tX.shape[0]

        lTrainLoss.append(epochLoss / numSamples)
        print(f'\rEpoch: {(ii + 1):3d} / {numEpochs}, Loss: {lTrainLoss[-1]:.6f}', end = '')

    print('')

    return oModel, lTrainLoss

def CalcModelLoss(oModel: nn.Module, dlData: DataLoader, hL: Callable, runDevice: torch.device) -> float:

    oModel.eval()
    valLoss    = 0.0
    numSamples = 0

    with torch.inference_mode():
        for tX, vY in dlData:
            tX = tX.to(runDevice)
            vY = vY.to(runDevice)
            valLoss    += tX.shape[0] * hL(oModel(tX), vY).item()
            numSamples += tX.shape[0]

    return valLoss / numSamples

def CalcAccuracy(oModel: nn.Module, dlData: DataLoader, runDevice: torch.device) -> float:

    numCorrect = 0
    numSamples = 0
    oModel.eval()

    with torch.inference_mode():
        for tX, vY in dlData:
            tX = tX.to(runDevice)
            vY = vY.to(runDevice)
            vYHat = torch.argmax(oModel(tX), dim = 1)
            numCorrect += torch.sum(vYHat == vY).item()
            numSamples += len(vY)

    return numCorrect / numSamples

def GenFilterNormalizedDirection(oModel: nn.Module, seedNum: int) -> Dict[str, torch.Tensor]:

    oGen = torch.Generator(device = 'cpu').manual_seed(seedNum)
    dDirection = {}

    for paramName, tParam in oModel.named_parameters():
        tDirection = torch.randn(tParam.shape, generator = oGen, dtype = tParam.dtype)
        if tParam.ndim > 1:
            tWeightNorm = torch.linalg.vector_norm(tParam.detach().cpu().flatten(start_dim = 1), dim = 1)
            tDirectNorm = torch.linalg.vector_norm(tDirection.flatten(start_dim = 1), dim = 1).clamp_min(1e-12)
            tScale = (tWeightNorm / tDirectNorm).reshape((-1,) + (1,) * (tParam.ndim - 1))
            tDirection *= tScale
        else:
            tDirection.zero_() #<! Do not perturb bias parameters

        dDirection[paramName] = tDirection.to(tParam.device)

    return dDirection

def CalcLossLandscape(oModel: nn.Module, dlData: DataLoader, hL: Callable, vAlpha: NDArray, vBeta: NDArray, runDevice: torch.device, seedNum: int) -> NDArray:

    dBase = {paramName: tParam.detach().clone() for paramName, tParam in oModel.named_parameters()}
    dDir1 = GenFilterNormalizedDirection(oModel, seedNum + 1)
    dDir2 = GenFilterNormalizedDirection(oModel, seedNum + 2)
    mLoss = np.empty((len(vBeta), len(vAlpha)))

    for ii, betaVal in enumerate(vBeta):
        for jj, alphaVal in enumerate(vAlpha):
            with torch.no_grad():
                for paramName, tParam in oModel.named_parameters():
                    tParam.copy_(dBase[paramName] + alphaVal * dDir1[paramName] + betaVal * dDir2[paramName])
            mLoss[ii, jj] = CalcModelLoss(oModel, dlData, hL, runDevice)

    with torch.no_grad():
        for paramName, tParam in oModel.named_parameters():
            tParam.copy_(dBase[paramName])

    return mLoss

## Skip Connection Effect

In [Visualizing the Loss Landscape of Neural Nets](https://arxiv.org/abs/1712.09913), Li et al. show that skip connections produce substantially smoother loss landscapes.

![](https://i.imgur.com/Yr6ShY3.png)

This notebook reproduces the qualitative effect with two small convolutional networks. Both networks have the same parameters and training recipe. The only difference is the identity addition

$$
\boldsymbol{x}_{l + 1} = \mathcal{F}(\boldsymbol{x}_{l}; \boldsymbol{\theta}_{l}) + \boldsymbol{x}_{l}.
$$

After training, the loss is evaluated on a two-dimensional, filter-normalized plane around each solution:

$$
f(\alpha, \beta) = \mathcal{L}(\boldsymbol{\theta}^{*} + \alpha \boldsymbol{d}_{1} + \beta \boldsymbol{d}_{2}).
$$

* <font color='brown'>(**#**)</font> This is a small qualitative reproduction, not the full ResNet / CIFAR experiment from the paper.

In [ ]:
# Parameters

# Data
testSize = 0.2

# Model
numBlocks   = 12
numChannels = 16
numCls      = 10

# Training
batchSize = 128
numEpochs = 30
learnRate = 2e-3

# Loss Landscape
numGridPts = 21
gridRadius = 1.0

## Generate / Load Data

In [ ]:
# Generate / Load Data

dDigits = load_digits()
tX = torch.tensor(dDigits.images[:, None, :, :], dtype = torch.float32) / 16.0
vY = torch.tensor(dDigits.target, dtype = torch.int64)

vIdxTrain, vIdxTest = train_test_split(np.arange(len(vY)), test_size = testSize, random_state = seedNum, stratify = vY)

print(f'Number of training samples: {len(vIdxTrain)}')
print(f'Number of test samples    : {len(vIdxTest)}')
print(f'Image dimensions          : {tuple(tX.shape[1:])}')

In [ ]:
# Data Sets and Data Loaders

dsTrain = TensorDataset(tX[vIdxTrain], vY[vIdxTrain])
dsTest  = TensorDataset(tX[vIdxTest],  vY[vIdxTest])

oDataGenPlain = torch.Generator().manual_seed(seedNum)
oDataGenSkip  = torch.Generator().manual_seed(seedNum)
dlTrainPlain  = DataLoader(dsTrain, batch_size = batchSize, shuffle = True, generator = oDataGenPlain)
dlTrainSkip   = DataLoader(dsTrain, batch_size = batchSize, shuffle = True, generator = oDataGenSkip)
dlTest        = DataLoader(dsTest, batch_size = 4 * batchSize, shuffle = False)

# A fixed training subset, without augmentation, as in the reference implementation
dsLandscape = TensorDataset(tX[vIdxTrain[:512]], vY[vIdxTrain[:512]])
dlLandscape = DataLoader(dsLandscape, batch_size = 4 * batchSize, shuffle = False)

In [ ]:
# Plot Samples

hF, vHa = plt.subplots(nrows = 2, ncols = 5, figsize = (10, 4))

for ii, hA in enumerate(vHa.flat):
    hA.imshow(tX[ii, 0], cmap = 'gray')
    hA.set_title(f'Label: {vY[ii].item()}')
    hA.set_axis_off()

hF.tight_layout()

## Train the Models

The two models use the same initialization, data order, optimizer and number of updates. Hence, the skip connection is the controlled variable in the experiment.

In [ ]:
# Model Realization

torch.manual_seed(seedNum) #<! Reproducibility
oModelPlain = SimpleConvNet(useSkip = False, numBlocks = numBlocks, numChannels = numChannels, numCls = numCls)

torch.manual_seed(seedNum) #<! Reproducibility
oModelSkip = SimpleConvNet(useSkip = True, numBlocks = numBlocks, numChannels = numChannels, numCls = numCls)

numParamsPlain = sum(tParam.numel() for tParam in oModelPlain.parameters())
numParamsSkip  = sum(tParam.numel() for tParam in oModelSkip.parameters())

print(f'Plain model parameters: {numParamsPlain:,}')
print(f'Skip model parameters : {numParamsSkip:,}')

In [ ]:
# Validate the Models

tXBatch, vYBatch = next(iter(dlTest))

with torch.inference_mode():
    mZPlain = oModelPlain(tXBatch)
    mZSkip  = oModelSkip(tXBatch)

print(f'Input dimensions       : {tuple(tXBatch.shape)}')
print(f'Plain output dimensions: {tuple(mZPlain.shape)}')
print(f'Skip output dimensions : {tuple(mZSkip.shape)}')

In [ ]:
# Run Device and Loss Function

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device
hL        = nn.CrossEntropyLoss().to(runDevice)

print(f'Run Device: {runDevice}')

In [ ]:
# Train the Plain Model

startTime = time.time()
oModelPlain, lLossPlain = TrainModel(oModelPlain, dlTrainPlain, hL, numEpochs, learnRate, runDevice)
print(f'Training time: {(time.time() - startTime):.2f} [Sec]')

In [ ]:
# Train the Model with Skip Connections

startTime = time.time()
oModelSkip, lLossSkip = TrainModel(oModelSkip, dlTrainSkip, hL, numEpochs, learnRate, runDevice)
print(f'Training time: {(time.time() - startTime):.2f} [Sec]')

### Training Analysis

The residual model is expected to optimize more easily as depth increases.  
The loss curves provide the first check before analyzing the geometry around the final solutions.

In [ ]:
# Plot Training Loss

hF, hA = plt.subplots(figsize = (8, 5))

hA.plot(lLossPlain, lw = 2, label = 'Without Skip Connections')
hA.plot(lLossSkip,  lw = 2, label = 'With Skip Connections')
hA.set_title('Training Loss')
hA.set_xlabel('Epoch')
hA.set_ylabel('Cross Entropy Loss')
hA.set_yscale('log')
hA.legend()
hA.grid(True)

In [ ]:
# Model Accuracy

print(f'Plain model test accuracy: {CalcAccuracy(oModelPlain, dlTest, runDevice):.2%}')
print(f'Skip model test accuracy : {CalcAccuracy(oModelSkip,  dlTest, runDevice):.2%}')

## Loss Landscape

The random directions are normalized independently for each convolutional filter, as proposed in the paper. Bias parameters are held fixed. This removes much of the scale ambiguity caused by neural networks.

The surface is evaluated on a fixed subset of the training data without augmentation. For visualization, the loss is clipped and shown on a logarithmic scale, following the reference implementation.

In [ ]:
# Calculate the Loss Landscapes

vAlpha = np.linspace(-gridRadius, gridRadius, numGridPts)
vBeta  = np.linspace(-gridRadius, gridRadius, numGridPts)
mAlpha, mBeta = np.meshgrid(vAlpha, vBeta)

startTime = time.time()
mLossPlain = CalcLossLandscape(oModelPlain, dlLandscape, hL, vAlpha, vBeta, runDevice, seedNum)
mLossSkip  = CalcLossLandscape(oModelSkip,  dlLandscape, hL, vAlpha, vBeta, runDevice, seedNum)
print(f'Landscape calculation time: {(time.time() - startTime):.2f} [Sec]')

assert mLossPlain.shape == mLossSkip.shape == (numGridPts, numGridPts)
assert np.all(np.isfinite(mLossPlain)) and np.all(np.isfinite(mLossSkip))

### The 2D View

The skip connected network reaches a deep low loss basin, whereas the equally deep plain network remains in a much higher loss region.  
This small experiment reproduces the optimization and landscape effect qualitatively.  
The exact ridges depend on model size, data and the sampled directions.

In [ ]:
# Plot the Loss Landscapes

mPlotPlain = np.log10(np.clip(mLossPlain, 1e-3, 10.0))
mPlotSkip  = np.log10(np.clip(mLossSkip,  1e-3, 10.0))
valMin     = min(np.min(mPlotPlain), np.min(mPlotSkip))
valMax     = max(np.max(mPlotPlain), np.max(mPlotSkip))

hF = plt.figure(figsize = (15, 6))

for ii, (mLoss, titleStr) in enumerate(((mPlotPlain, 'Without Skip Connections'), (mPlotSkip, 'With Skip Connections'))):
    hA = hF.add_subplot(1, 2, ii + 1, projection = '3d')
    hSurf = hA.plot_surface(mAlpha, mBeta, mLoss, cmap = cm.coolwarm, vmin = valMin, vmax = valMax, linewidth = 0, antialiased = True)
    hA.set_title(titleStr)
    hA.set_xlabel(r'$\alpha$')
    hA.set_ylabel(r'$\beta$')
    hA.set_zlabel(r'$\log_{10}(\mathrm{Loss})$')
    hA.view_init(elev = 28, azim = -55)

_ = hF.colorbar(hSurf, ax = hF.axes, shrink = 0.65, pad = 0.08, label = r'$\log_{10}(\mathrm{Loss})$')

* <font color='red'>(**?**)</font> What happens to the two landscapes as `numBlocks` increases?
* <font color='green'>(**@**)</font> Repeat the experiment with several random direction seeds.  
  A 2D plane is only one view of the high dimensional loss surface.